# 🎬 SHORT MAKER V3 — FULL SCREEN & VOCAB HIGHLIGHT

Tạo video Short kể chuyện 1 phút dạy tiếng Anh viral (Tự động 100%).
- **Lõi V1 (Cực kỳ ổn định)**: Lấy nguyên bản cách phân chia câu và đọc của V1 để không bao giờ bị treo.
- **Hình ảnh V3**: Full màn hình dọc 9:16, Zoom nhẹ mượt mà.
- **Phụ đề V3**: File `.ass` nền đen, chữ trắng to rõ, các từ vựng nổi bật sẽ được đổi màu vàng.

In [ ]:
# @title ⚙️ CELL 1: CÀI ĐẶT (chạy 1 lần, ~3 phút)
import os, subprocess, sys

print('⏳ Đang cài đặt thư viện...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'qwen-tts', 'huggingface_hub', 'pydub', 'openai-whisper',
    'pysrt', 'requests', 'playwright', 'nest_asyncio'], check=True,
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

os.system('apt-get install -y -qq ffmpeg sox libsox-fmt-all 2>/dev/null')

from IPython.display import clear_output
clear_output()

import torch, soundfile, whisper, pysrt, requests, json, re, time, shutil
from pathlib import Path
from qwen_tts import Qwen3TTSModel

print('✅ TẤT CẢ THƯ VIỆN ĐÃ SẴN SÀNG!')
print(f'   GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print('   -> Upload file giọng mẫu (.wav) lên cột trái 📁')
print('   -> Chạy Cell 2 để cấu hình')

In [ ]:
# @title 📝 CELL 2: CẤU HÌNH
# @markdown ---
# @markdown ### Chủ đề video
topic = 'A funny story about a sleepy office worker trying to fix the WiFi router by hitting it' # @param {type:'string'}
# @markdown ---
# @markdown ### API Keys
gemini_api_key = '' # @param {type:'string'}
# @markdown ---
# @markdown ### Kết nối TurboFlow (Tạo ảnh)
bridge_url = '' # @param {type:'string'}
# @markdown ---
# @markdown ### 🎤 Giọng đọc (Voice Clone)
file_giong_mau = 'yo.wav' # @param {type:'string'}
loi_thoai_giong_mau = 'Wait, have you ever noticed that time feels faster as we get older? It is kind of scary, right? But actually, there is a hidden logic behind it.' # @param {type:'string'}

assert topic.strip(), '❌ Nhập chủ đề video!'
assert gemini_api_key.strip(), '❌ Nhập Gemini API key!'
assert bridge_url.strip(), '❌ Nhập Bridge URL!'

slug = re.sub(r'[^a-z0-9]+', '_', topic.lower()).strip('_')
print(f'✅ Cấu hình OK! Project: {slug}')
print(f'   → Chạy Cell 3 để tạo video')

In [ ]:
# @title 🚀 CELL 3: TẠO VIDEO (chạy 1 lần, ~5-10 phút)
import gc, base64
from IPython.display import Audio, display, HTML
from pydub import AudioSegment

P = Path(f'/content/{slug}')
P.mkdir(parents=True, exist_ok=True)
(P / 'images').mkdir(exist_ok=True)

print(f"{'='*60}")
print(f'🎬 STORY MAKER V3: {topic}')
print(f"{'='*60}")

# ==========================================
# STEP 1: GENERATE SCRIPT (Y CHANG V1)
# ==========================================
print(f'\n📖 [1/5] Sinh kịch bản (Gemini AI)...')

PROMPT = '''You are a top-tier viral YouTube Shorts scriptwriter and English language educator. Write an engaging English learning/storytelling script about: "{topic}"

## CRITICAL LENGTH & DURATION REQUIREMENT:
- The script MUST be around 150 words (strictly between 140 and 155 words). This is a hard requirement.
- At normal speaking pace, 150 words = approximately 50 seconds of audio.
- Write in a flowing, storytelling narrative — NOT bullet points.
- Use clear, professional, yet conversational English, making it perfect for English learners to listen and study.
- To hit exactly ~150 words, please structure the script length paragraph by paragraph as follows:
  * Paragraph 1 (HOOK): ~25 words (1-2 sentences).
  * Paragraph 2 (BODY 1): ~30 words (2-3 sentences).
  * Paragraph 3 (BODY 2): ~30 words (2-3 sentences).
  * Paragraph 4 (BODY 3): ~35 words (2-3 sentences).
  * Paragraph 5 (CTA): ~30 words (2 sentences).
  * Total Target: 150 words.
- You MUST count the words in your generated script before outputting. If it is less than 140 words or more than 155 words, rewrite and adjust it to fit the range.

## PUNCTUATION RULES (TTS system reads these as timing cues):
- COMMAS (,) = chain actions smoothly in one breath, no pause. Use these to connect flowing descriptions.
- PERIODS (.) QUESTION MARKS (?) EXCLAMATION (!) SEMICOLONS (;) COLONS (:) = end of sentence, brief 0.1s pause.
- BLANK LINES between paragraphs = dramatic 0.35s pause for emphasis.
- NEVER use ellipsis (...). Use a period or new paragraph instead.
- Do NOT include any markdown bold (* or **) or italic inside the script text.

## VOCABULARY TO TEACH (visual_keywords):
Extract exactly 10-12 useful English vocabulary words or short phrases that appear literally in YOUR script, in strict order of appearance.
- These words will be HIGHLIGHTED in the video to teach English learners.
- "keyword": The exact word or phrase as it appears in the script.
- "search_query": A concise prompt for an AI image generator to illustrate this word. MUST end with the EXACT phrase: ', 9:16 vertical, vibrant 2D flat illustration style, consistent character'.

## OUTPUT (valid JSON only, no markdown):
{{
  "word_count": 150,
  "script": "Full script here with blank lines between paragraphs.",
  "visual_keywords": [
    {{"keyword": "word1", "search_query": "... , 9:16 vertical, vibrant 2D flat illustration style, consistent character"}}
  ]
}}'''

messages = [{'role': 'system', 'content': 'Return valid JSON only with keys: word_count, script, and visual_keywords. You MUST strictly follow the word count limits and constraints. Make sure the generated script has around 150 words.'},
            {'role': 'user', 'content': PROMPT.format(topic=topic)}]

max_retries = 3
script_data = {}
for attempt in range(max_retries):
    try:
        gemini_contents = []
        system_instruction = None
        for msg in messages:
            if msg['role'] == 'system':
                system_instruction = {'parts': [{'text': msg['content']}]}
            else:
                role = 'user' if msg['role'] == 'user' else 'model'
                gemini_contents.append({'role': role, 'parts': [{'text': msg['content']}]})
        
        body = {
            'contents': gemini_contents,
            'generationConfig': {'responseMimeType': 'application/json', 'temperature': 0.7}
        }
        if system_instruction:
            body['systemInstruction'] = system_instruction
        
        resp = requests.post(
            f'https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent?key={gemini_api_key.strip()}',
            headers={'Content-Type': 'application/json'},
            json=body,
            timeout=30)
        resp.raise_for_status()
        raw = resp.json()['candidates'][0]['content']['parts'][0]['text'].strip()
        raw = re.sub(r'^```json\s*', '', raw); raw = re.sub(r'\s*```$', '', raw)
        script_data = json.loads(raw)
        
        script_text = script_data.get('script', '')
        word_count = len(script_text.split())
        print(f'   Generated script word count: {word_count} words (Attempt {attempt+1}/{max_retries})')
        
        if 130 <= word_count <= 165:
            print('   ✅ Word count is within target range!')
            break
        else:
            if attempt < max_retries - 1:
                messages.append({'role': 'assistant', 'content': raw})
                messages.append({
                    'role': 'user',
                    'content': f'The previous script has {word_count} words. Rewrite it to be strictly between 140 and 155 words. Keep the JSON format.'
                })
    except Exception as e:
        if attempt >= max_retries - 1: raise
        time.sleep(5)

script_text = script_data.get('script', '')
keywords = script_data.get('visual_keywords', [])
(P / 'script.txt').write_text(script_text, encoding='utf-8')
(P / 'keywords.json').write_text(json.dumps(script_data, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'   ✅ {len(script_text.split())} từ, {len(keywords)} từ vựng cần dạy.')

# ==========================================
# STEP 2: GET IMAGES (TurboFlow)
# ==========================================
print(f'\n🖼️ [2/5] Tải ảnh từ TurboFlow...')
burl = bridge_url.strip().rstrip('/')
health_resp = requests.get(f'{burl}/health', timeout=5)
assert health_resp.ok, f'❌ Không kết nối được bridge!'
server_time = health_resp.json().get('time', time.time())

prompts = [kw['search_query'] for kw in keywords]
txt_content = '\n\n'.join(prompts)
(P / 'prompts.txt').write_text(txt_content, encoding='utf-8')
Path('/content/prompts.txt').write_text(txt_content, encoding='utf-8')
print(f'   📝 Đã tạo /content/prompts.txt')
print('   => Mở TurboFlow Extension, copy nội dung file này vào và bấm Start (chọn Aspect Ratio 9:16).')

expected_count = len(prompts)
downloaded_imgs = []
since_ts = server_time - 10

for attempt in range(1200):
    time.sleep(2)
    try: 
        imgs_resp = requests.get(f'{burl}/images', params={'since': since_ts}, timeout=10)
        if imgs_resp.ok:
            all_items = imgs_resp.json().get('items', [])
            new_items = [it for it in all_items if it['name'] not in [x['name'] for x in downloaded_imgs]]
            if new_items:
                for it in new_items:
                    img_path = P / 'images' / f'img_{len(downloaded_imgs):02d}.jpg'
                    img_data = requests.get(f'{burl}/download', params={'name': it['name']}, timeout=30)
                    img_path.write_bytes(img_data.content)
                    downloaded_imgs.append({'name': it['name'], 'path': img_path})
                    print(f'\r      Đã tải: {img_path.name} ({len(downloaded_imgs)}/{expected_count})', end='', flush=True)
                if len(downloaded_imgs) >= expected_count:
                    print('\n   ✅ Đã tải đủ ảnh!')
                    break
            elif attempt % 15 == 0:
                print(f'\r      ⏳ Đang chờ ảnh ({len(downloaded_imgs)}/{expected_count})...', end='', flush=True)
    except Exception as e:
        pass
print()
assert len(downloaded_imgs) >= expected_count, f'❌ Hết thời gian chờ!'

# ==========================================
# STEP 3: GENERATE VOICE (Y CHANG V1 100%)
# ==========================================
print(f'\n🎤 [3/5] Tạo giọng đọc (Voice Clone từ {file_giong_mau})...')

assert os.path.exists(file_giong_mau), f'❌ Chưa thấy file giọng mẫu!'
tts_model = Qwen3TTSModel.from_pretrained('Qwen/Qwen3-TTS-12Hz-1.7B-Base', torch_dtype=torch.float16,
    device_map='cuda:0', attn_implementation='sdpa')

print('   ⚡️ Đang học giọng mẫu...')
clone_prompt = tts_model.create_voice_clone_prompt(
    ref_audio=file_giong_mau, ref_text=loi_thoai_giong_mau.strip(), x_vector_only_mode=False)

nghi_ngan = AudioSegment.silent(duration=300)  # 0.3s giữa các câu
nghi_dai = AudioSegment.silent(duration=500)   # 0.5s giữa các đoạn
final_audio = AudioSegment.silent(duration=300) # 0.3s mở đầu

raw_text = script_text.replace('\r', '')
paragraphs = re.split(r'\n\s*\n', raw_text)
total_sentences = sum(len(re.split(r'(?<=[.?!;:\n])\s+', p.strip())) for p in paragraphs if p.strip())
line_count = 0

print(f'   🚀 Thu âm {total_sentences} câu...')
for p_text in paragraphs:
    p_text = p_text.strip()
    if not p_text: continue
    sentences = re.split(r'(?<=[.?!;:\n])\s+', p_text)
    for s in sentences:
        s = s.strip()
        if not s: continue
        line_count += 1
        print(f'   🎙️ [{line_count}/{total_sentences}]: {s[:60]}...')
        with torch.inference_mode():
            w, sr = tts_model.generate_voice_clone(text=s, voice_clone_prompt=clone_prompt)
        soundfile.write('/content/temp_line.wav', w[0], sr)
        final_audio += AudioSegment.from_wav('/content/temp_line.wav') + nghi_ngan
        if os.path.exists('/content/temp_line.wav'): os.remove('/content/temp_line.wav')
    final_audio += nghi_dai

mp3_path = str(P / 'audio.mp3')
final_audio.export(mp3_path, format='mp3')
del tts_model, clone_prompt; gc.collect(); torch.cuda.empty_cache()
print(f'   ✅ audio.mp3 đã tạo')
display(Audio(mp3_path, autoplay=False))

# ==========================================
# STEP 4: WORD-LEVEL SUBTITLES (.ASS) V3 STYLE
# ==========================================
print(f'\n📝 [4/5] Tạo phụ đề từng từ (Whisper)...')
w_model = whisper.load_model('base.en')
result = w_model.transcribe(mp3_path, word_timestamps=True, language='en')
del w_model; gc.collect(); torch.cuda.empty_cache()

ass_header = """[Script Info]
ScriptType: v4.00+
PlayResX: 1080
PlayResY: 1920
WrapStyle: 1

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: Default,Arial,65,&H00FFFFFF,&H000000FF,&H00000000,&H80000000,-1,0,0,0,100,100,0,0,3,10,0,2,50,50,150,1

[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
"""
def srt_time(s):
    h = int(s // 3600); m = int((s % 3600) // 60); sec = int(s % 60); ms = int((s % 1) * 100)
    return f"{h}:{m:02d}:{sec:02d}.{ms:02d}"

ass_events = []
target_words = [kw['keyword'].lower() for kw in keywords]
for segment in result['segments']:
    start_t = srt_time(segment['start'])
    end_t = srt_time(segment['end'])
    text = segment['text'].strip()
    # Tô màu vàng các từ khóa
    for tw in target_words:
        pattern = re.compile(r'\b' + re.escape(tw) + r'\b', re.IGNORECASE)
        text = pattern.sub(lambda m: f"{{\\c&H00FFFF&\\b1}}{m.group(0)}{{\\c&HFFFFFF&\\b0}}", text)
    ass_events.append(f"Dialogue: 0,{start_t},{end_t},Default,,0,0,0,,{text}")

ass_path = P / 'subtitles.ass'
ass_path.write_text(ass_header + '\n'.join(ass_events), encoding='utf-8')
print('   ✅ Đã tạo file subtitles.ass (phụ đề nổi bật từ vựng)')

# ==========================================
# STEP 5: RENDER VIDEO FULL MÀN HÌNH (V3 STYLE)
# ==========================================
print(f'\n🎬 [5/5] Render video (Zoom Ảnh Mượt Mà & Ghép Phụ Đề)...')
audio_dur = len(AudioSegment.from_mp3(mp3_path)) / 1000.0
img_dur = audio_dur / len(downloaded_imgs)
list_txt = P / 'inputs.txt'
lines = []
for i, img in enumerate(downloaded_imgs):
    # Zoom rat nhe vao giua (zoom tu 1.0 toi max khoang 1.1)
    pan_filter = "zoompan=z='min(zoom+0.0005,1.1)':x='iw/2-(iw/zoom)/2':y='ih/2-(ih/zoom)/2':d={dur_frames}:s=1080x1920"
    dur_frames = int(img_dur * 30)
    out_vid = P / f'clip_{i:02d}.mp4'
    cmd = ['ffmpeg', '-y', '-loop', '1', '-i', str(img['path']), '-vf', pan_filter.format(dur_frames=dur_frames),
           '-c:v', 'libx264', '-t', str(img_dur), '-pix_fmt', 'yuv420p', '-r', '30', str(out_vid)]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    lines.append(f"file '{out_vid.name}'")
list_txt.write_text('\n'.join(lines))

merged_vid = P / 'merged.mp4'
subprocess.run(['ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', str(list_txt),
                '-c', 'copy', str(merged_vid)], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

final_output = P / 'final_short.mp4'
cmd = ['ffmpeg', '-y', '-i', str(merged_vid), '-i', mp3_path, 
       '-vf', f"ass='{ass_path}'",
       '-c:v', 'libx264', '-preset', 'fast', '-crf', '18',
       '-c:a', 'aac', '-b:a', '192k', '-shortest', str(final_output)]
subprocess.run(cmd, capture_output=True)
size_mb = final_output.stat().st_size / (1024*1024)
print(f'\n{"="*60}')
print(f'VIDEO HOÀN TẤT! ({size_mb:.1f} MB)')
print(f'   {final_output}')
print(f'{"="*60}')


In [ ]:
# @title 📥 CELL 4: TẢI VIDEO VỀ MÁY
from google.colab import files
files.download(str(final_output))
print('Đang tải video...')